# Instacart Lakehouse Analytics — Gold Business Tables

## Objective

This notebook transforms the cleaned Silver-layer data into business-ready Gold tables.

The Gold layer aggregates transaction data into customer, product, department, and order-behaviour summaries that support dashboarding and business analysis.

## Inputs

- `silver_orders`
- `silver_product_dimension`
- `silver_order_items`

## Outputs

- `gold_customer_summary`
- `gold_product_performance`
- `gold_department_summary`
- `gold_order_behavior`

## 1. Load Silver Tables

Load the cleaned Silver Delta tables from Unity Catalog for aggregation and business-metric creation.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_orders = spark.table("workspace.default.silver_orders")
silver_products = spark.table("workspace.default.silver_product_dimension")
silver_items = spark.table("workspace.default.silver_order_items")

## 2. Create an Order-Level Summary

Aggregate the transaction-level Silver data to one record per order.

This table calculates basket size, unique products, and order-level reorder rate and is used as the foundation for customer and time-based analysis.

In [0]:
# Aggregate line items into one record per order
order_level = (
    silver_items
    .groupBy(
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "eval_set"
    )
    .agg(
        F.count("*").alias("basket_size"),
        F.countDistinct("product_id").alias("unique_products"),
        F.avg("reordered").alias("order_reorder_rate")
    )
)

In [0]:
print("Order-level rows:", order_level.count())
display(order_level.limit(20))

Order-level rows: 109467


order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,eval_set,basket_size,unique_products,order_reorder_rate
12,152610,22,6,8,10.0,prior,15,15,0.4
18,118860,3,4,20,6.0,prior,28,28,0.35714285714285715
67,5738,4,3,12,6.0,prior,8,8,0.75
70,128213,2,6,15,4.0,prior,4,4,0.25
93,130239,15,6,10,19.0,prior,14,14,0.5714285714285714
161,88526,11,1,23,19.0,prior,9,9,0.7777777777777778
186,167393,1,0,16,null,prior,9,9,0.0
190,189157,3,1,13,30.0,prior,11,11,0.09090909090909091
225,130657,9,1,9,22.0,prior,11,11,0.9090909090909091
261,153543,14,0,13,9.0,prior,10,10,0.8


## 3. Build Customer-Level Metrics

Create customer summaries from both transaction-level and order-level data.

The metrics include:

- sampled order count
- total items purchased
- unique products purchased
- average basket size
- largest basket size
- average reorder rate
- average days between orders

Because the project uses sampled prior-order data, these customer measures are described as sample-based rather than complete lifetime metrics.

In [0]:
customer_item_metrics = (
    silver_items
    .groupBy("user_id")
    .agg(
        F.count("*").alias("total_items_purchased"),
        F.countDistinct("product_id").alias("unique_products_purchased"),
        F.avg("reordered").alias("customer_reorder_rate")
    )
)

customer_order_metrics = (
    order_level
    .groupBy("user_id")
    .agg(
        F.countDistinct("order_id").alias("sampled_order_count"),
        F.avg("basket_size").alias("average_basket_size"),
        F.max("basket_size").alias("largest_basket_size"),
        F.avg("days_since_prior_order").alias("average_days_between_orders")
    )
)

## 4. Identify Each Customer's Favourite Department

Count purchases by customer and department, then use a window function to select the most frequently purchased department for each customer.

Alphabetical ordering is used as a deterministic tie-breaker.

In [0]:
# Rank departments within each customer by purchase frequency
department_counts = (
    silver_items
    .groupBy("user_id", "department")
    .agg(F.count("*").alias("department_purchase_count"))
)

department_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("department_purchase_count"),
        F.asc("department")
    )
)

favorite_department = (
    department_counts
    .withColumn("department_rank", F.row_number().over(department_window))
    .filter(F.col("department_rank") == 1)
    .select(
        "user_id",
        F.col("department").alias("favorite_department")
    )
)

## 5. Create the Gold Customer Summary

Combine order-level metrics, item-level metrics, and favourite-department attributes into one customer-level business table.

In [0]:
gold_customer_summary = (
    customer_order_metrics
    .join(customer_item_metrics, on="user_id", how="left")
    .join(favorite_department, on="user_id", how="left")
)

## 6. Create Product Performance Metrics

Aggregate the enriched order-item data by product to measure purchase volume, customer reach, reorder behaviour, and average cart position.

In [0]:
# Summarize product demand and reorder performance
gold_product_performance = (
    silver_items
    .groupBy(
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department"
    )
    .agg(
        F.count("*").alias("total_purchases"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.countDistinct("user_id").alias("unique_customers"),
        F.sum("reordered").alias("total_reorders"),
        F.avg("reordered").alias("reorder_rate"),
        F.avg("add_to_cart_order").alias("average_cart_position")
    )
)

## 7. Create Department Performance Metrics

Aggregate transaction data by department to measure item volume, order reach, customer reach, product breadth, reorder rate, and average cart position.

In [0]:
gold_department_summary = (
    silver_items
    .groupBy(
        "department_id",
        "department"
    )
    .agg(
        F.count("*").alias("total_items_purchased"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.countDistinct("user_id").alias("unique_customers"),
        F.countDistinct("product_id").alias("unique_products"),
        F.avg("reordered").alias("reorder_rate"),
        F.avg("add_to_cart_order").alias("average_cart_position")
    )
)

## 8. Create Ordering-Behaviour Metrics

Summarize order activity by day of week and hour of day.

These metrics support analysis of shopping patterns, peak ordering periods, basket size, and reorder behaviour.

In [0]:
gold_order_behavior = (
    order_level
    .groupBy(
        "order_dow",
        "order_hour_of_day"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("user_id").alias("unique_customers"),
        F.avg("basket_size").alias("average_basket_size"),
        F.avg("order_reorder_rate").alias("average_reorder_rate")
    )
)

## 9. Write Gold Delta Tables

Store the business-ready aggregates as managed Delta tables in Unity Catalog.

Overwrite mode is used to keep the portfolio pipeline repeatable during development.

In [0]:
# Persist the Gold layer for dashboarding and analysis
gold_tables = {
    "gold_customer_summary": gold_customer_summary,
    "gold_product_performance": gold_product_performance,
    "gold_department_summary": gold_department_summary,
    "gold_order_behavior": gold_order_behavior
}

for table_name, df in gold_tables.items():
    full_table_name = f"workspace.default.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Created: {full_table_name}")

Created: workspace.default.gold_customer_summary
Created: workspace.default.gold_product_performance
Created: workspace.default.gold_department_summary
Created: workspace.default.gold_order_behavior


## 10. Validate Gold Outputs

Confirm that each Gold table was created successfully and inspect sample results for completeness and business usability.

In [0]:
for table_name in gold_tables:
    full_table_name = f"workspace.default.{table_name}"
    count = spark.table(full_table_name).count()
    print(f"{table_name}: {count:,} rows")

gold_customer_summary: 54,726 rows
gold_product_performance: 36,117 rows
gold_department_summary: 21 rows
gold_order_behavior: 168 rows


In [0]:
display(
    spark.table("workspace.default.gold_product_performance")
    .orderBy(F.desc("total_purchases"))
    .limit(20)
)

product_id,product_name,aisle_id,aisle,department_id,department,total_purchases,unique_orders,unique_customers,total_reorders,reorder_rate,average_cart_position
24852,Banana,24,fresh fruits,4,produce,16495,16495,11821,14325,0.8684449833282814,4.864504395271294
13176,Bag of Organic Bananas,24,fresh fruits,4,produce,13605,13605,9976,11728,0.8620360161705255,4.98970966556413
21137,Organic Strawberries,24,fresh fruits,4,produce,9544,9544,7726,7719,0.8087803855825649,7.544530595138307
21903,Organic Baby Spinach,123,packaged vegetables fruits,4,produce,8369,8369,6757,6830,0.8161070617756004,7.528139562671765
47209,Organic Hass Avocado,24,fresh fruits,4,produce,7227,7227,5720,5936,0.8213643282136432,6.82122595821226
47766,Organic Avocado,24,fresh fruits,4,produce,6061,6061,4750,4941,0.8152120112192708,6.396799208051477
47626,Large Lemon,24,fresh fruits,4,produce,5720,5720,4888,4236,0.7405594405594406,8.066783216783216
16797,Strawberries,24,fresh fruits,4,produce,5048,5048,4238,3782,0.7492076069730587,7.242670364500793
27966,Organic Raspberries,123,packaged vegetables fruits,4,produce,4926,4926,4010,3868,0.785221274868047,7.293747462444173
26209,Limes,24,fresh fruits,4,produce,4898,4898,4226,3550,0.7247856267864434,8.751735402204982


## Gold Layer Summary

The Silver-layer data was transformed into four business-ready Gold tables.

### Tables created

- `gold_customer_summary`
- `gold_product_performance`
- `gold_department_summary`
- `gold_order_behavior`

These tables provide the core analytical layer for customer segmentation, product and department performance, order-pattern analysis, and dashboard development.

The next notebook implements data-quality checks across the Silver and Gold layers.